### PSD Pipeline
#### Table of Contents:
* [1. Environment Setup](#1-environment-setup)
    * [1.1 Library Imports](#11-library-imports)
    * [1.2 Dataset loading/inspection](#12-dataset-loadinginspection)
* [2. Data Segmentation](#data-segmentation)
* [3. Feature Extraction and Class Distribution](#3-feature-extraction--class-distribution)
    * [3.1 PSD](#31-psd-feature-extraction)
    * [3.2 Class Distribution](#32-class-distribution)
* [4. Classification Schemes](#4-classification-schemes)
    * [4.1 Emotional vs. Neutral](#41-emotional-vs-neutral-remap--class-distribution)
    * [4.2 Positive vs. Negative](#42-positive-vs-negative-remap--class-distribution)
* [5. Model Training](#5-model-training)
    * [5.1 XGBoost](#51-xgboost)
        * [Emotional vs. Neutral](#511-emotional-vs-neutral)
        * [Positive vs. Negative](#512-positive-vs-negative)




### 1. Environment Setup

##### 1.1 Library Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import welch

import matplotlib.pyplot as plt
import seaborn as sns

import optuna
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, balanced_accuracy_score, classification_report


/Users/connor/miniconda3/envs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### 1.2 Dataset loading/inspection

In [2]:
dataset_path = Path('DEED')
eeg_dataset = []
subject_ids = []

for file in dataset_path.iterdir():
    mat = loadmat(file)
    eeg = mat['Data']
    fname = file.stem  
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    subject_part = [part for part in fname.split("_") if part.startswith("S")][0]
    label = int(label_part[1:])  
    subject_id = subject_part[1:-1]  # last two digits = subject number
    
    eeg_dataset.append((eeg, label))
    subject_ids.append(subject_id)

print(f"Loaded {len(eeg_dataset)} trials.")
print(f"Unique subjects: {len(set(subject_ids))}")
print(f"Subject IDs: {sorted(set(subject_ids))}")
print("Example shapes:", [(arr.shape, lbl) for arr, lbl in eeg_dataset[:3]])

print("\nSample filename → label mapping:")
for file, (_, label) in zip(dataset_path.iterdir(), eeg_dataset[:10]):
    print(f"  {file.stem} → E{label}")

Loaded 533 trials.
Unique subjects: 34
Subject IDs: ['002', '003', '004', '005', '007', '011', '012', '013', '014', '015', '016', '017', '018', '020', '021', '022', '023', '024', '025', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '042']
Example shapes: [((6, 290000), 2), ((6, 36000), 2), ((6, 51000), 3)]

Sample filename → label mapping:
  G_S0321_M1_E2_R1_N2_raw_ref → E2
  G_S0213_M3_E2_R7_N2_raw_ref → E2
  G_S0393_M2_E3_R5_REM_raw_ref → E3
  G_S0243_M3_E2_R2_N2_raw_ref → E2
  G_S0311_M3_E5_R2_N2_raw_ref → E5
  G_S0031_M1_E3_R4_nan_raw_ref → E3
  G_S0043_M2_E2_R5_N2_raw_ref → E2
  G_S0072_M1_E0_R11_N1_raw_ref → E0
  G_S0242_M1_E3_R3_W_raw_ref → E3
  G_S0342_M2_E3_R3_N2_raw_ref → E3


### 2. Data Segmentation

Done into 20s windows.

In [3]:
def segmentation(eeg_dataset, subject_ids, window_sec, fs):
    window_size = int(window_sec * fs)
    X = []
    y = []
    groups = []

    for (eeg_array, label), sid in zip(eeg_dataset, subject_ids):
        n_samples = eeg_array.shape[1]
        start = 0
        while start + window_size <= n_samples:
            window = eeg_array[:, start:start + window_size]
            X.append(window)
            y.append(label)
            groups.append(sid)
            start += window_size  

    return X, y, groups

windows, window_labels, window_groups = segmentation(eeg_dataset, subject_ids, 20, 200)
print(f"Total 20 second windows: {len(windows)}")
print(f"Unique subjects: {len(set(window_groups))}")

Total 20 second windows: 7490
Unique subjects: 34


### 3. Feature Extraction & Class Distribution

PSD features are extracted from each 20s window using Welch's method. For each channel, log band power 
and relative band power are computed across the five standard frequency bands (delta, theta, alpha, beta, 
gamma), alongside time-domain mean and variance. Windows are then remapped into two binary classification schemes: Emotional vs. Neutral and Positive vs. Negative.

##### 3.1 PSD Feature Extraction

In [4]:
def extract_psd_features(segmented_windows, labels, fs=200):
    freq_bands = {
        'delta': (0.5, 4),
            'theta': (4, 8),
            'alpha': (8, 12),
            'beta': (12, 30),
            'gamma': (30, 45)
    }
    
    windows = np.array(segmented_windows)
    n_windows, n_channels, n_samples = windows.shape
    
    nperseg = min(512, n_samples)  
    noverlap = nperseg // 2
    
    all_features = []
    
    for ch_idx in range(n_channels):
        ch_data = windows[:, ch_idx, :]
        f, Pxx = welch(ch_data, fs=fs, nperseg=nperseg, noverlap=noverlap, axis=1)
        
        # Band power per band
        ch_band_powers = []
        for band_name, (low, high) in freq_bands.items():
            idx = np.logical_and(f >= low, f <= high)
            band_power = np.trapz(Pxx[:, idx], f[idx], axis=1) 
            ch_band_powers.append(band_power)
        ch_band_powers = np.column_stack(ch_band_powers) 

        # Log band power
        log_band_power = np.log(ch_band_powers + 1e-10)
        all_features.append(log_band_power)

        # Relative band power
        total_power = ch_band_powers.sum(axis=1, keepdims=True)
        relative_band_power = ch_band_powers / (total_power + 1e-10)
        all_features.append(relative_band_power)

        # Time-domain features
        ch_mean = np.mean(ch_data, axis=1, keepdims=True)
        ch_var = np.var(ch_data, axis=1, keepdims=True)
        all_features.append(ch_mean)
        all_features.append(ch_var)

        
    
    
    X = np.hstack(all_features)
    y = np.array(labels)
    
    return X, y

##### 3.2 Class Distribution

In [5]:
X, y = extract_psd_features(windows, window_labels)
print(X.shape)
print(y.shape) 

print("\n=== Total Class Distribution===")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"E{label}: {count} windows ({count/len(y)*100:.1f}%)")

(7490, 72)
(7490,)

=== Total Class Distribution===
E0: 1294 windows (17.3%)
E1: 305 windows (4.1%)
E2: 1069 windows (14.3%)
E3: 2855 windows (38.1%)
E4: 1698 windows (22.7%)
E5: 269 windows (3.6%)


### 4. Classification Schemes

##### 4.1 Emotional vs. Neutral (remap + class distribution)  
Emotional {E1, E2, E4, E5} vs. Neutral Dream {E3} 

In [6]:
def remap_emotional_neutral(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 3, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [1, 2, 4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_en, mask_en = remap_emotional_neutral(y)
X_en = X[mask_en]
groups_en = np.array(window_groups)[mask_en]

print(f"\n=== Emotional vs. Neutral Class Distribution ===")
print(f"  Total: {len(y_en)}")
print(f"  Neutral (0): {np.sum(y_en == 0)} ({np.sum(y_en == 0)/len(y_en)*100:.1f}%)")
print(f"  Emotional (1): {np.sum(y_en == 1)} ({np.sum(y_en == 1)/len(y_en)*100:.1f}%)")


=== Emotional vs. Neutral Class Distribution ===
  Total: 6196
  Neutral (0): 2855 (46.1%)
  Emotional (1): 3341 (53.9%)


In [7]:
# Subject Wise Class Distribution (EN)
print("=== Subject Wise Class Distribution (EN) ===")
for subject in sorted(set(groups_en)):
    mask = groups_en == subject
    labels = y_en[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

=== Subject Wise Class Distribution (EN) ===
Subject 002: 185 windows, classes: [0 1], counts: [ 18 167]
Subject 003: 400 windows, classes: [0 1], counts: [188 212]
Subject 004: 245 windows, classes: [0 1], counts: [104 141]
Subject 005: 242 windows, classes: [0 1], counts: [155  87]
Subject 007: 511 windows, classes: [0 1], counts: [ 78 433]
Subject 011: 43 windows, classes: [0], counts: [43]
Subject 012: 95 windows, classes: [0 1], counts: [83 12]
Subject 013: 26 windows, classes: [0 1], counts: [12 14]
Subject 014: 56 windows, classes: [0], counts: [56]
Subject 015: 269 windows, classes: [0 1], counts: [122 147]
Subject 016: 213 windows, classes: [0 1], counts: [184  29]
Subject 017: 213 windows, classes: [0 1], counts: [177  36]
Subject 018: 7 windows, classes: [0], counts: [7]
Subject 020: 105 windows, classes: [0 1], counts: [95 10]
Subject 021: 375 windows, classes: [0 1], counts: [162 213]
Subject 022: 207 windows, classes: [0 1], counts: [112  95]
Subject 023: 173 windows, cla

##### 4.2 Positive vs. Negative (remap + class distribution)
Positive {E4, E5} vs. Negative {E1, E2}

In [8]:
def remap_positive_negative(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_pn, mask_pn = remap_positive_negative(y)
X_pn = X[mask_pn]
groups_pn = np.array(window_groups)[mask_pn]

print(f"\n=== Positive vs. Negative Class Distribution ===")
print(f"  Total: {len(y_pn)}")
print(f"  Negative (0): {np.sum(y_pn == 0)} ({np.sum(y_pn == 0)/len(y_pn)*100:.1f}%)")
print(f"  Positive (1): {np.sum(y_pn == 1)} ({np.sum(y_pn == 1)/len(y_pn)*100:.1f}%)")


=== Positive vs. Negative Class Distribution ===
  Total: 3341
  Negative (0): 1374 (41.1%)
  Positive (1): 1967 (58.9%)


In [9]:
# Subject Wise Class Distribution (PN)
print("=== Subject Wise Class Distribution (PN) ===")
for subject in sorted(set(groups_pn)):
    mask = groups_pn == subject
    labels = y_pn[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

=== Subject Wise Class Distribution (PN) ===
Subject 002: 167 windows, classes: [0 1], counts: [89 78]
Subject 003: 212 windows, classes: [0 1], counts: [ 65 147]
Subject 004: 141 windows, classes: [0 1], counts: [68 73]
Subject 005: 87 windows, classes: [0 1], counts: [33 54]
Subject 007: 433 windows, classes: [0 1], counts: [206 227]
Subject 012: 12 windows, classes: [0], counts: [12]
Subject 013: 14 windows, classes: [1], counts: [ 0 14]
Subject 015: 147 windows, classes: [0 1], counts: [ 22 125]
Subject 016: 29 windows, classes: [1], counts: [ 0 29]
Subject 017: 36 windows, classes: [0 1], counts: [28  8]
Subject 020: 10 windows, classes: [0], counts: [10]
Subject 021: 213 windows, classes: [0 1], counts: [ 28 185]
Subject 022: 95 windows, classes: [0 1], counts: [46 49]
Subject 023: 100 windows, classes: [0 1], counts: [21 79]
Subject 024: 235 windows, classes: [0 1], counts: [139  96]
Subject 025: 34 windows, classes: [1], counts: [ 0 34]
Subject 027: 26 windows, classes: [0 1], 

### 5. Model Training
For both models and classification schemes, hyperparameters are optimized using Optuna with 5-fold StratifiedGroupKFold on the full dataset. The resulting best parameters are fixed and used for final evaluation with LOSO. For comparison, performance is also assessed using standard 10-fold cross-validation.

##### General Functions for Training

In [10]:
optuna.logging.set_verbosity(optuna.logging.INFO)

def hyperparameter_training(X, y, groups, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
            'max_depth':        trial.suggest_int('max_depth', 3, 8),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma':            trial.suggest_float('gamma', 0, 5),
            'eval_metric':      'logloss',
        }
        cv = StratifiedGroupKFold(n_splits=5)
        scores = []
        for train_idx, val_idx in cv.split(X, y, groups):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            neg = np.sum(y_train == 0)
            pos = np.sum(y_train == 1)
            params['scale_pos_weight'] = neg/pos

            model = XGBClassifier(**params, n_jobs=-1, random_state = 42)
            model.fit(X_train, y_train)
            preds = model.predict(X_val)

            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials = 50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")


    # Save best params
    best_params = study.best_params
    best_params['random_state'] = 42
    best_params['n_jobs'] = -1
    return best_params

In [11]:
def loso_loop(X, y, groups, params, model_name, classification_scheme):
    # LOSO evaluation with fixed params
    print(f"\n=== LOSO - {model_name} ({classification_scheme}) ===")
    logo = LeaveOneGroupOut()
    accs = []
    f1s = []
    aurocs = []
    bal_accs = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        if len(np.unique(y[test_idx])) < 2:
            continue

        subject = groups[test_idx[0]]
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        #Per fold class weighting
        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)

        fold_params = params.copy()
        fold_params['scale_pos_weight'] = neg/pos

        model = XGBClassifier(**fold_params)
        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))
        print(f"Subject {subject} | Balanced Accuracy: {bal_accs[-1]:.4f} | Accuracy: {accs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUROC: {aurocs[-1]:.4f}")

    print(f"Balanced Accuracy:  {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"Accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"AUROC:  {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")


In [ ]:
def ten_fold_cv_loop(X, y, params, model_name, classification_scheme):
    print(f"\n=== 10-Fold CV - {model_name} ({classification_scheme}) ===")
    # 10 Fold Cross CV with same tuned params
    cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    accs = []
    f1s = []
    bal_accs = []
    aurocs = []

    for train_idx, test_idx in cv_10fold.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)
        fold_params = params.copy()
        fold_params['scale_pos_weight'] = neg / pos

        model = XGBClassifier(**fold_params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
        # importance = model.feature_importances_ 
        # print(len(importance))

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))

    print(f"Accuracy:          {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:                {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"Balanced Accuracy: {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"AUROC:             {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")

#### 5.1 XGBoost

##### 5.1.1 Emotional vs. Neutral

In [13]:
#Hyperparameter tuning
xgb_en_params = hyperparameter_training(X_en, y_en, groups_en, "XGBoost", "Emotional vs. Neutral")

[I 2026-03-02 16:51:41,111] A new study created in memory with name: no-name-a95188ac-b90f-4b85-98ab-6696a829a57c


=== Hyperparameter Tuning - XGBoost (Emotional vs. Neutral) ===


[I 2026-03-02 16:51:59,233] Trial 0 finished with value: 0.47414551405111804 and parameters: {'n_estimators': 592, 'max_depth': 8, 'learning_rate': 0.07144053047557743, 'subsample': 0.9713893832016589, 'colsample_bytree': 0.7682223373222768, 'min_child_weight': 4, 'gamma': 2.0008184599205343}. Best is trial 0 with value: 0.47414551405111804.
[I 2026-03-02 16:52:21,068] Trial 1 finished with value: 0.4598018101638705 and parameters: {'n_estimators': 245, 'max_depth': 7, 'learning_rate': 0.07140351427388074, 'subsample': 0.6085774214571149, 'colsample_bytree': 0.7541038761720772, 'min_child_weight': 1, 'gamma': 0.9079243080633326}. Best is trial 0 with value: 0.47414551405111804.
[I 2026-03-02 16:52:28,895] Trial 2 finished with value: 0.47743700215291085 and parameters: {'n_estimators': 533, 'max_depth': 7, 'learning_rate': 0.06445682819719564, 'subsample': 0.9458099539429139, 'colsample_bytree': 0.6956776831707663, 'min_child_weight': 5, 'gamma': 3.3800205733300386}. Best is trial 2 wi

Best params: {'n_estimators': 191, 'max_depth': 6, 'learning_rate': 0.18147460162233034, 'subsample': 0.7549220803067248, 'colsample_bytree': 0.8509174588333758, 'min_child_weight': 9, 'gamma': 0.8351118528727512}
Best CV F1: 0.5013


In [14]:
#LOSO Evaluation
xgb_loso = loso_loop(X_en, y_en, groups_en, xgb_en_params, "XGBoost", "Emotional vs. Neutral")


=== LOSO - XGBoost (Emotional vs. Neutral) ===
Subject 002 | Balanced Accuracy: 0.4930 | Accuracy: 0.6216 | F1: 0.4516 | AUROC: 0.4232
Subject 003 | Balanced Accuracy: 0.4293 | Accuracy: 0.4225 | F1: 0.4180 | AUROC: 0.4306
Subject 004 | Balanced Accuracy: 0.4519 | Accuracy: 0.4490 | F1: 0.4477 | AUROC: 0.4416
Subject 005 | Balanced Accuracy: 0.5471 | Accuracy: 0.5620 | F1: 0.5425 | AUROC: 0.5626
Subject 007 | Balanced Accuracy: 0.5099 | Accuracy: 0.4188 | F1: 0.3883 | AUROC: 0.5117
Subject 012 | Balanced Accuracy: 0.6596 | Accuracy: 0.7789 | F1: 0.6149 | AUROC: 0.7279
Subject 013 | Balanced Accuracy: 0.4107 | Accuracy: 0.3846 | F1: 0.3203 | AUROC: 0.3512
Subject 015 | Balanced Accuracy: 0.3721 | Accuracy: 0.3755 | F1: 0.3723 | AUROC: 0.3444
Subject 016 | Balanced Accuracy: 0.4994 | Accuracy: 0.5869 | F1: 0.4608 | AUROC: 0.5553
Subject 017 | Balanced Accuracy: 0.4327 | Accuracy: 0.3146 | F1: 0.3065 | AUROC: 0.3192
Subject 020 | Balanced Accuracy: 0.5868 | Accuracy: 0.4952 | F1: 0.4192 

In [15]:
#10Fold CV
xgb_10f = ten_fold_cv_loop(X_en, y_en, xgb_en_params, "XGBoost", "Emotional vs. Neutral")


=== 10-Fold CV - XGBoost (Emotional vs. Neutral) ===
72
72
72
72
72
72
72
72
72
72
Accuracy:          0.6911 ± 0.0119
F1:                0.6887 ± 0.0122
Balanced Accuracy: 0.6886 ± 0.0123
AUROC:             0.7426 ± 0.0124


##### 5.1.2 Positive vs. Negative

In [16]:
#Hyperparameter tuning
xgb_pn_params = hyperparameter_training(X_pn, y_pn, groups_pn, "XGBoost", "Positive vs. Negative")

[I 2026-03-02 16:59:52,489] A new study created in memory with name: no-name-7a7426cc-c01c-4347-89a5-91a7fdfd9c6f


=== Hyperparameter Tuning - XGBoost (Positive vs. Negative) ===


[I 2026-03-02 16:59:54,310] Trial 0 finished with value: 0.5189536846996855 and parameters: {'n_estimators': 167, 'max_depth': 3, 'learning_rate': 0.02741829805081102, 'subsample': 0.6973661298778068, 'colsample_bytree': 0.6319693513746643, 'min_child_weight': 2, 'gamma': 1.4615401615328767}. Best is trial 0 with value: 0.5189536846996855.
[I 2026-03-02 17:00:02,127] Trial 1 finished with value: 0.5147532419721405 and parameters: {'n_estimators': 217, 'max_depth': 8, 'learning_rate': 0.012486937860442983, 'subsample': 0.9983163851370626, 'colsample_bytree': 0.9690578454038967, 'min_child_weight': 6, 'gamma': 1.5896384874011305}. Best is trial 0 with value: 0.5189536846996855.
[I 2026-03-02 17:00:03,358] Trial 2 finished with value: 0.5205516360904008 and parameters: {'n_estimators': 159, 'max_depth': 6, 'learning_rate': 0.21365293463539264, 'subsample': 0.8399212460530686, 'colsample_bytree': 0.6618895022076053, 'min_child_weight': 7, 'gamma': 2.752423108192022}. Best is trial 2 with v

Best params: {'n_estimators': 407, 'max_depth': 8, 'learning_rate': 0.01402581521465742, 'subsample': 0.626564597369804, 'colsample_bytree': 0.9344612645758967, 'min_child_weight': 10, 'gamma': 4.507096172027978}
Best CV F1: 0.5425


In [17]:
#LOSO Evaluation
xgb_loso = loso_loop(X_pn, y_pn, groups_pn, xgb_pn_params, "XGBoost", "Positive vs. Negative")


=== LOSO - XGBoost (Positive vs. Negative) ===
Subject 002 | Balanced Accuracy: 0.5334 | Accuracy: 0.5449 | F1: 0.5240 | AUROC: 0.6322
Subject 003 | Balanced Accuracy: 0.4205 | Accuracy: 0.5236 | F1: 0.4160 | AUROC: 0.3535
Subject 004 | Balanced Accuracy: 0.6289 | Accuracy: 0.6241 | F1: 0.6192 | AUROC: 0.6992
Subject 005 | Balanced Accuracy: 0.4655 | Accuracy: 0.4828 | F1: 0.4647 | AUROC: 0.4675
Subject 007 | Balanced Accuracy: 0.5045 | Accuracy: 0.5012 | F1: 0.5002 | AUROC: 0.5066
Subject 015 | Balanced Accuracy: 0.5215 | Accuracy: 0.8231 | F1: 0.5174 | AUROC: 0.2018
Subject 017 | Balanced Accuracy: 0.7589 | Accuracy: 0.6944 | F1: 0.6630 | AUROC: 0.7812
Subject 021 | Balanced Accuracy: 0.5120 | Accuracy: 0.7840 | F1: 0.5122 | AUROC: 0.5826
Subject 022 | Balanced Accuracy: 0.5783 | Accuracy: 0.5684 | F1: 0.5274 | AUROC: 0.6553
Subject 023 | Balanced Accuracy: 0.3924 | Accuracy: 0.6200 | F1: 0.3827 | AUROC: 0.3773
Subject 024 | Balanced Accuracy: 0.4118 | Accuracy: 0.4128 | F1: 0.4089 

In [18]:
#10Fold CV
xgb_10f = ten_fold_cv_loop(X_pn, y_pn, xgb_pn_params, "XGBoost", "Positive vs. Negative")


=== 10-Fold CV - XGBoost (Positive vs. Negative) ===
72
72
72
72
72
72
72
72
72
72
Accuracy:          0.6756 ± 0.0248
F1:                0.6723 ± 0.0240
Balanced Accuracy: 0.6795 ± 0.0232
AUROC:             0.7440 ± 0.0254
